# Lasso Topic Selection for Market Returns
Using `mret` from macro.csv, fit Lasso across penalty parameters on the topics data, select the penalty yielding 5 non-zero coefficients, then run OLS.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, lasso_path
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Load data
macro = pd.read_csv('macro.csv', parse_dates=['date'])
topics = pd.read_csv('topics.csv', parse_dates=['date'])

# Merge on date
df = pd.merge(macro, topics, on='date')
topic_cols = [c for c in topics.columns if c != 'date']

print(f"Merged dataset: {df.shape[0]} observations, {len(topic_cols)} topic features")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

# Standardize topics for Lasso
X = df[topic_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
def lasso_select_and_ols(y, X_scaled, topic_cols, df, target_nonzero=5):
    """Fit Lasso across alphas, select penalty for target_nonzero coefficients, run OLS."""
    alphas_grid = np.logspace(-6, 1, 200)
    alphas_out, coefs, _ = lasso_path(X_scaled, y, alphas=alphas_grid, max_iter=50000)
    n_nonzero = np.sum(coefs != 0, axis=0)

    mask_5 = n_nonzero == target_nonzero
    if mask_5.any():
        alpha_sel = alphas_out[mask_5].min()
        idx = np.where(alphas_out == alpha_sel)[0][0]
    else:
        idx = np.argmin(np.abs(n_nonzero - target_nonzero))
        alpha_sel = alphas_out[idx]

    selected_mask = coefs[:, idx] != 0
    selected_topics = [topic_cols[j] for j in range(len(topic_cols)) if selected_mask[j]]

    X_ols = sm.add_constant(df[selected_topics])
    ols_model = sm.OLS(y, X_ols).fit()

    return alpha_sel, selected_topics, ols_model

In [ ]:
# === Part 1: mret (market return) ===
y_mret = df['mret'].values
alpha_mret, topics_mret, ols_mret = lasso_select_and_ols(y_mret, X_scaled, topic_cols, df)

print("=== mret (Market Return) ===")
print(f"Alpha = {alpha_mret:.6f}, R2 = {ols_mret.rsquared:.4f}, Adj R2 = {ols_mret.rsquared_adj:.4f}")
print(ols_mret.summary())

In [ ]:
# === Part 2: vol, indpro, indprol1 ===
for outcome in ['vol', 'indpro', 'indprol1']:
    y = df[outcome].values
    alpha_sel, sel_topics, ols = lasso_select_and_ols(y, X_scaled, topic_cols, df)
    print(f"\n{'='*80}")
    print(f"=== {outcome} ===")
    print(f"Alpha = {alpha_sel:.6f}, R2 = {ols.rsquared:.4f}, Adj R2 = {ols.rsquared_adj:.4f}")
    print(f"Selected topics: {sel_topics}")
    for t in sel_topics:
        sig = '***' if ols.pvalues[t]<0.01 else '**' if ols.pvalues[t]<0.05 else '*' if ols.pvalues[t]<0.1 else ''
        print(f"  {t:40s}  coef={ols.params[t]:+12.4f}  p={ols.pvalues[t]:.4f} {sig}")
    print(ols.summary())

In [ ]:
# === Part 3: All industry volatility columns ===
vol_cols = [c for c in macro.columns if c.endswith('_vol')]
all_results = []

for outcome in vol_cols:
    y = df[outcome].values
    alpha_sel, sel_topics, ols = lasso_select_and_ols(y, X_scaled, topic_cols, df)
    all_results.append({
        'outcome': outcome, 'r2': ols.rsquared, 'adj_r2': ols.rsquared_adj,
        'topics': sel_topics, 'n': len(sel_topics)
    })

# Summary table
print(f"{'Outcome':<15} {'R2':>7} {'AdjR2':>7} {'#':>3}   Selected Topics")
print("=" * 110)
for r in all_results:
    print(f"{r['outcome']:<15} {r['r2']:7.4f} {r['adj_r2']:7.4f} {r['n']:3d}   {', '.join(r['topics'])}")

r2_vals = [r['r2'] for r in all_results]
print(f"\nIndustry volatilities: Mean R2 = {np.mean(r2_vals):.4f}, "
      f"Median = {np.median(r2_vals):.4f}, Min = {np.min(r2_vals):.4f}, Max = {np.max(r2_vals):.4f}")

In [ ]:
# === Most frequently selected topics across ALL outcomes ===
from collections import Counter
all_topics_flat = []
for r in all_results:
    all_topics_flat.extend(r['topics'])
# Also include the macro outcome topics
for outcome in ['mret', 'vol', 'indpro', 'indprol1']:
    y = df[outcome].values
    _, sel_t, _ = lasso_select_and_ols(y, X_scaled, topic_cols, df)
    all_topics_flat.extend(sel_t)

freq = Counter(all_topics_flat)
print("Most frequently selected topics across all 52 outcomes:")
for t, count in freq.most_common(20):
    print(f"  {t:40s}  {count:2d}/52")

## Part (c): Forecasting `indprol1` with RBF Kernel + Ridge (Best Model from HW1)

Chronological train/test split to simulate real-time forecasting.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import RidgeCV, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# ── 1. Data Preparation ─────────────────────────────────────────────────────
macro = pd.read_csv('macro.csv', parse_dates=['date'])
topics = pd.read_csv('topics.csv', parse_dates=['date'])
df_c = pd.merge(macro, topics, on='date').sort_values('date').reset_index(drop=True)

topic_cols = [c for c in topics.columns if c != 'date']
X_all = df_c[topic_cols]
y_all = df_c['indprol1']

# Drop rows where y is NaN
mask = y_all.notna()
X_all = X_all[mask].reset_index(drop=True)
y_all = y_all[mask].reset_index(drop=True)
dates  = df_c.loc[mask, 'date'].reset_index(drop=True)

print(f"After dropping NaN: {len(y_all)} observations")
print(f"Date range: {dates.iloc[0].date()} to {dates.iloc[-1].date()}")

# Chronological 80/20 split
split = int(len(y_all) * 0.80)
X_train, X_test = X_all.iloc[:split], X_all.iloc[split:]
y_train, y_test = y_all.iloc[:split], y_all.iloc[split:]

print(f"Train: {split} obs  ({dates.iloc[0].date()} – {dates.iloc[split-1].date()})")
print(f"Test:  {len(y_all)-split} obs  ({dates.iloc[split].date()} – {dates.iloc[-1].date()})")

# Compute gamma='scale': 1 / (n_features * X_train_scaled.var())
_scaler = StandardScaler().fit(X_train)
gamma_scale = 1.0 / (X_train.shape[1] * _scaler.transform(X_train).var())
print(f"gamma (scale): {gamma_scale:.6f}")

In [ ]:
# ── 2. Modeling Pipeline ─────────────────────────────────────────────────────
pipe = Pipeline([
    ('scaler',  StandardScaler()),
    ('rbf',     Nystroem(gamma=gamma_scale, n_components=1000, random_state=42)),
    ('ridge',   RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5)),
])

pipe.fit(X_train, y_train)

# ── 3. Evaluation ───────────────────────────────────────────────────────────
y_pred = pipe.predict(X_test)
oos_r2  = r2_score(y_test, y_pred)
oos_mse = mean_squared_error(y_test, y_pred)
best_alpha = pipe.named_steps['ridge'].alpha_

print("=" * 60)
print("  RBF Kernel + Ridge  —  Out-of-Sample Results")
print("=" * 60)
print(f"  Best Ridge alpha (CV): {best_alpha:.4f}")
print(f"  OOS R-squared:         {oos_r2:.4f}")
print(f"  OOS MSE:               {oos_mse:.6f}")
print("=" * 60)

# Baselines for comparison
pipe_ols = Pipeline([('scaler', StandardScaler()), ('lr', LinearRegression())])
pipe_ols.fit(X_train, y_train)
print(f"\n  Baseline OLS OOS R2:     {r2_score(y_test, pipe_ols.predict(X_test)):.4f}")

pipe_ridge_lin = Pipeline([('scaler', StandardScaler()),
                           ('ridge', RidgeCV(alphas=np.logspace(-3,3,50), cv=5))])
pipe_ridge_lin.fit(X_train, y_train)
print(f"  Linear Ridge OOS R2:     {r2_score(y_test, pipe_ridge_lin.predict(X_test)):.4f}")

In [ ]:
# ── 4. Reasoning for Modeling Decisions ──────────────────────────────────────
reasoning = """
REASONING FOR MODELING DECISIONS
================================

1. RBF Kernel Approximation (Nystroem, n_components=1000)
   The Radial Basis Function kernel implicitly maps the 180 topic features into
   a much higher-dimensional space where non-linear interactions between topics
   become linear relationships. For example, the joint effect of "Recession" and
   "Oil market" attention spiking simultaneously may differ from the sum of their
   individual effects — the RBF kernel captures these interaction and threshold
   effects without manually engineering them. Nystroem provides a tractable
   finite-dimensional approximation (1000 random features) of the full kernel
   matrix, making it scalable to our 400-observation dataset.

2. Ridge Regression (L2 penalty) over Lasso (L1)
   After the RBF expansion, we have 1000 highly correlated random features.
   Lasso tends to arbitrarily select one among correlated features and zero out
   the rest, which is unstable in this setting. Ridge shrinks all coefficients
   proportionally, distributing weight across correlated features and producing
   more stable predictions. With 1000 features and ~320 training observations,
   strong regularization is essential — RidgeCV with 5-fold CV automatically
   selects the penalty (alpha) that minimizes prediction error.

3. Chronological Train/Test Split (80/20)
   A random train/test split would allow the model to "peek" at future data when
   predicting past observations, inflating OOS performance. Since we are
   forecasting industrial production growth one period ahead (indprol1), a
   chronological split is essential: the model trains only on historical data and
   is evaluated on genuinely future observations. This simulates a real-time
   forecasting environment where a policymaker or investor would only have access
   to past news topic attention when forming their forecast.
"""
print(reasoning)

## Part (d): Build a Document-Term Matrix from WSJ Headlines

Using `CountVectorizer` on the `headline` column of `articles.pq`.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Load headlines
articles = pd.read_parquet('articles.pq')
print(f"Articles: {articles.shape[0]} headlines")
print(f"Date range: {articles['display_date'].min().date()} to {articles['display_date'].max().date()}")
print(f"\nSample headlines:")
for h in articles['headline'].head(5):
    print(f"  {h}")

# Build document-term matrix
vectorizer = CountVectorizer(stop_words='english')
dtm = vectorizer.fit_transform(articles['headline'])

feature_names = vectorizer.get_feature_names_out()

print(f"\nDocument-Term Matrix shape: {dtm.shape}")
print(f"  {dtm.shape[0]} documents (headlines)")
print(f"  {dtm.shape[1]} unique terms (after removing English stop words)")
print(f"  Non-zero entries: {dtm.nnz} ({100*dtm.nnz/(dtm.shape[0]*dtm.shape[1]):.4f}% dense)")
print(f"  Avg terms per headline: {dtm.sum(axis=1).mean():.1f}")

In [ ]:
# Most frequent terms in the corpus
term_freq = np.array(dtm.sum(axis=0)).flatten()
top_idx = term_freq.argsort()[::-1][:25]

print("Top 25 most frequent terms:")
for i, idx in enumerate(top_idx, 1):
    print(f"  {i:2d}. {feature_names[idx]:20s}  freq={term_freq[idx]}")

# Store as a DataFrame for downstream use
dtm_df = pd.DataFrame(dtm.toarray(), columns=feature_names)
dtm_df.insert(0, 'display_date', articles['display_date'].values)
print(f"\nDTM DataFrame shape: {dtm_df.shape}")
dtm_df.head()

## Part (e): Lasso on Counts vs. Topics — Comparative Analysis

Repeat the Lasso → OLS procedure from parts (a)/(b) using monthly word counts instead of topics. Compare R2 and determine how many count features are needed to match the topics R2.

In [ ]:
# ── Aggregate article-level DTM to monthly word counts ───────────────────────
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import lasso_path

articles = pd.read_parquet('articles.pq')
macro = pd.read_csv('macro.csv', parse_dates=['date'])
topics = pd.read_csv('topics.csv', parse_dates=['date'])

vec = CountVectorizer(stop_words='english')
dtm = vec.fit_transform(articles['headline'])
feat_names = vec.get_feature_names_out()

# Prefix to avoid column name collisions (e.g. "date" is a vocab word)
count_cols = ['w_' + f for f in feat_names]
display_name = {c: c[2:] for c in count_cols}

articles['date'] = articles['display_date'].dt.to_period('M').dt.to_timestamp()
dtm_df = pd.DataFrame(dtm.toarray(), columns=count_cols)
dtm_df['date'] = articles['date'].values
dtm_monthly = dtm_df.groupby('date').sum().reset_index()

df_e = pd.merge(macro, dtm_monthly, on='date')
df_topics_e = pd.merge(macro, topics, on='date')
topic_cols_e = [c for c in topics.columns if c != 'date']

X_counts = df_e[count_cols].values.astype(float)
scaler_c = StandardScaler()
X_counts_scaled = scaler_c.fit_transform(X_counts)

X_topics_e = df_topics_e[topic_cols_e].values
scaler_te = StandardScaler()
X_topics_scaled_e = scaler_te.fit_transform(X_topics_e)

print(f"Monthly counts: {df_e.shape[0]} months x {len(count_cols)} terms")
print(f"Topics:         {df_topics_e.shape[0]} months x {len(topic_cols_e)} topics")

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────────
def lasso_k_ols(y, X_sc, cols, df_use, k=5):
    """Lasso path -> select k non-zero -> OLS R2."""
    a_grid = np.logspace(-6, 1, 200)
    a_out, coefs, _ = lasso_path(X_sc, y, alphas=a_grid, max_iter=50000)
    nnz = np.sum(coefs != 0, axis=0)
    m = nnz == k
    idx = np.where(a_out == a_out[m].min())[0][0] if m.any() else np.argmin(np.abs(nnz - k))
    sel = [cols[j] for j in range(len(cols)) if coefs[j, idx] != 0]
    if not sel:
        return [], 0.0
    ols = sm.OLS(y, sm.add_constant(df_use[sel])).fit()
    return sel, ols.rsquared

def find_k_match(y, X_sc, cols, df_use, target_r2):
    """Smallest k whose OLS R2 >= target_r2."""
    a_grid = np.logspace(-7, 1, 300)
    a_out, coefs, _ = lasso_path(X_sc, y, alphas=a_grid, max_iter=50000)
    nnz = np.sum(coefs != 0, axis=0)
    for k in sorted(set(nnz)):
        if k == 0: continue
        m = nnz == k
        if not m.any(): continue
        idx = np.where(a_out == a_out[m].min())[0][0]
        sel = [cols[j] for j in range(len(cols)) if coefs[j, idx] != 0]
        if not sel: continue
        ols = sm.OLS(y, sm.add_constant(df_use[sel])).fit()
        if ols.rsquared >= target_r2:
            return k, ols.rsquared
    return None, None

print("Helper functions defined.")

In [ ]:
# ── Run comparison for all outcomes ──────────────────────────────────────────
vol_cols_macro = [c for c in macro.columns if c.endswith('_vol')]
outcome_vars = ['mret', 'vol', 'indpro', 'indprol1'] + vol_cols_macro

print(f"{'Outcome':<15} {'Topics R2':>10} {'Counts R2':>10}  {'k to match':>10} {'Matched R2':>10}   Count terms (k=5)")
print("=" * 130)

topics_r2s, counts_r2s, k_matches = [], [], []

for outcome in outcome_vars:
    y_t = df_topics_e[outcome].values
    y_c = df_e[outcome].values

    t_names, t_r2 = lasso_k_ols(y_t, X_topics_scaled_e, topic_cols_e, df_topics_e, k=5)
    c_names, c_r2 = lasso_k_ols(y_c, X_counts_scaled, count_cols, df_e, k=5)
    k_m, m_r2 = find_k_match(y_c, X_counts_scaled, count_cols, df_e, t_r2)

    topics_r2s.append(t_r2)
    counts_r2s.append(c_r2)
    k_matches.append(k_m)

    c_disp = ", ".join(display_name[c] for c in c_names[:5]) if c_names else "N/A"
    k_str = str(k_m) if k_m else ">max"
    m_str = f"{m_r2:.4f}" if m_r2 else "N/A"
    print(f"{outcome:<15} {t_r2:10.4f} {c_r2:10.4f}  {k_str:>10} {m_str:>10}   {c_disp}")

In [ ]:
# ── Summary ─────────────────────────────────────────────────────────────────
topics_r2s = np.array(topics_r2s)
counts_r2s = np.array(counts_r2s)
k_valid = [k for k in k_matches if k is not None]

print("=" * 70)
print("SUMMARY: Topics (180 features) vs. Counts (18,093 terms)")
print("=" * 70)
print(f"\nWith k=5 features each:")
print(f"  Topics  — mean R2: {topics_r2s.mean():.4f}  median: {np.median(topics_r2s):.4f}")
print(f"  Counts  — mean R2: {counts_r2s.mean():.4f}  median: {np.median(counts_r2s):.4f}")
print(f"  Topics beat Counts: {(topics_r2s > counts_r2s).sum()}/{len(outcome_vars)} outcomes")

print(f"\nTo match Topics R2 using counts:")
print(f"  Mean k needed:   {np.mean(k_valid):.1f}")
print(f"  Median k needed: {np.median(k_valid):.1f}")
print(f"  Range:           {np.min(k_valid)} – {np.max(k_valid)}")
unmatched = sum(1 for k in k_matches if k is None)
print(f"  Could not match: {unmatched}/{len(outcome_vars)} outcomes")

## Part (f): Forecasting `indprol1` with Raw Word Counts vs. Topics

Compare the best count-based model to the topics model from part (c). Since X is now ~18K dimensions with only ~400 observations, severe regularization is critical and kernel expansion is avoided.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.metrics import r2_score, mean_squared_error

# ── 1. Data Preparation (counts) ────────────────────────────────────────────
articles_f = pd.read_parquet('articles.pq')
macro_f = pd.read_csv('macro.csv', parse_dates=['date'])
topics_f = pd.read_csv('topics.csv', parse_dates=['date'])

vec_f = CountVectorizer(stop_words='english')
dtm_f = vec_f.fit_transform(articles_f['headline'])
feat_f = vec_f.get_feature_names_out()
ccols = ['w_' + f for f in feat_f]

articles_f['date'] = articles_f['display_date'].dt.to_period('M').dt.to_timestamp()
dtm_df_f = pd.DataFrame(dtm_f.toarray(), columns=ccols)
dtm_df_f['date'] = articles_f['date'].values
dtm_mon = dtm_df_f.groupby('date').sum().reset_index()

df_f = pd.merge(macro_f[['date','indprol1']], dtm_mon, on='date').sort_values('date').reset_index(drop=True)

X_c = df_f[ccols]
y_c = df_f['indprol1']
dates_f = df_f['date']
mask_f = y_c.notna()
X_c, y_c, dates_f = X_c[mask_f].reset_index(drop=True), y_c[mask_f].reset_index(drop=True), dates_f[mask_f].reset_index(drop=True)

split_f = int(len(y_c) * 0.80)
Xtr_c, Xte_c = X_c.iloc[:split_f], X_c.iloc[split_f:]
ytr_c, yte_c = y_c.iloc[:split_f], y_c.iloc[split_f:]

print(f"Counts: {len(y_c)} obs, {X_c.shape[1]} features")
print(f"Train: {split_f}  ({dates_f.iloc[0].date()} – {dates_f.iloc[split_f-1].date()})")
print(f"Test:  {len(y_c)-split_f}  ({dates_f.iloc[split_f].date()} – {dates_f.iloc[-1].date()})")

In [ ]:
# ── 2. Count-based models ────────────────────────────────────────────────────
# Ridge
pipe_ridge_c = Pipeline([
    ('scaler', StandardScaler(with_mean=False)),
    ('ridge', RidgeCV(alphas=np.logspace(-1, 4, 50), cv=5)),
])
pipe_ridge_c.fit(Xtr_c, ytr_c)
pred_ridge_c = pipe_ridge_c.predict(Xte_c)

# Lasso
pipe_lasso_c = Pipeline([
    ('scaler', StandardScaler(with_mean=False)),
    ('lasso', LassoCV(cv=5, max_iter=10000)),
])
pipe_lasso_c.fit(Xtr_c, ytr_c)
pred_lasso_c = pipe_lasso_c.predict(Xte_c)
lasso_nnz = np.sum(pipe_lasso_c.named_steps['lasso'].coef_ != 0)

# ── 3. Topics baselines (same split) ────────────────────────────────────────
df_tf = pd.merge(macro_f[['date','indprol1']], topics_f, on='date').sort_values('date').reset_index(drop=True)
tcols = [c for c in topics_f.columns if c != 'date']
X_tf, y_tf = df_tf[tcols], df_tf['indprol1']
m_tf = y_tf.notna()
X_tf, y_tf = X_tf[m_tf].reset_index(drop=True), y_tf[m_tf].reset_index(drop=True)
sp_t = int(len(y_tf) * 0.80)

pipe_ridge_t = Pipeline([('scaler', StandardScaler()),
                         ('ridge', RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5))])
pipe_ridge_t.fit(X_tf.iloc[:sp_t], y_tf.iloc[:sp_t])

pipe_lasso_t = Pipeline([('scaler', StandardScaler()),
                         ('lasso', LassoCV(cv=5, max_iter=10000))])
pipe_lasso_t.fit(X_tf.iloc[:sp_t], y_tf.iloc[:sp_t])
lasso_nnz_t = np.sum(pipe_lasso_t.named_steps['lasso'].coef_ != 0)

# ── 4. Print results ────────────────────────────────────────────────────────
print("=" * 65)
print("  COMPARISON: Counts vs. Topics — indprol1 OOS Forecasting")
print("=" * 65)
rows = [
    ("Ridge (topics, p=180)",
     r2_score(y_tf.iloc[sp_t:], pipe_ridge_t.predict(X_tf.iloc[sp_t:])),
     mean_squared_error(y_tf.iloc[sp_t:], pipe_ridge_t.predict(X_tf.iloc[sp_t:])),
     180),
    ("Lasso (topics, p=180)",
     r2_score(y_tf.iloc[sp_t:], pipe_lasso_t.predict(X_tf.iloc[sp_t:])),
     mean_squared_error(y_tf.iloc[sp_t:], pipe_lasso_t.predict(X_tf.iloc[sp_t:])),
     lasso_nnz_t),
    ("Ridge (counts, p=18K)",
     r2_score(yte_c, pred_ridge_c),
     mean_squared_error(yte_c, pred_ridge_c),
     len(ccols)),
    ("Lasso (counts, p=18K)",
     r2_score(yte_c, pred_lasso_c),
     mean_squared_error(yte_c, pred_lasso_c),
     lasso_nnz),
]
print(f"  {'Model':<25} {'OOS R2':>10} {'OOS MSE':>12} {'NNZ/p':>8}")
print(f"  {'-'*25} {'-'*10} {'-'*12} {'-'*8}")
for name, r2, mse, nnz in rows:
    print(f"  {name:<25} {r2:10.4f} {mse:12.6f} {nnz:8d}")
print("=" * 65)

# Lasso-selected words
if lasso_nnz > 0:
    coefs_l = pipe_lasso_c.named_steps['lasso'].coef_
    nz_idx = np.where(coefs_l != 0)[0]
    nz_words = [feat_f[i] for i in nz_idx]
    nz_coefs = coefs_l[nz_idx]
    order = np.argsort(np.abs(nz_coefs))[::-1]
    print(f"\n  Lasso selected {lasso_nnz} word(s):")
    for i in order[:20]:
        print(f"    {nz_words[i]:20s}  coef={nz_coefs[i]:+.6f}")

## Part (g): TF-IDF vs. Raw Counts -- Contemporaneous Comparison

Convert text to TF-IDF, repeat the Lasso (k=5) + OLS exercise, and compare which terms each approach selects.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# ── Build monthly TF-IDF matrix ─────────────────────────────────────────────
articles_g = pd.read_parquet('articles.pq')
macro_g = pd.read_csv('macro.csv', parse_dates=['date'])
articles_g['date'] = articles_g['display_date'].dt.to_period('M').dt.to_timestamp()

tfidf_vec = TfidfVectorizer(stop_words='english')
tfidf_mat = tfidf_vec.fit_transform(articles_g['headline'])
tfidf_names = tfidf_vec.get_feature_names_out()

tfidf_cols = ['t_' + f for f in tfidf_names]
tfidf_df = pd.DataFrame(tfidf_mat.toarray(), columns=tfidf_cols)
tfidf_df['date'] = articles_g['date'].values
tfidf_monthly = tfidf_df.groupby('date').sum().reset_index()

df_tfidf = pd.merge(macro_g, tfidf_monthly, on='date')
X_tfidf = df_tfidf[tfidf_cols].values.astype(float)
sc_tfidf = StandardScaler()
X_tfidf_sc = sc_tfidf.fit_transform(X_tfidf)

# ── Build monthly raw counts matrix ─────────────────────────────────────────
count_vec_g = CountVectorizer(stop_words='english')
count_mat_g = count_vec_g.fit_transform(articles_g['headline'])
count_names_g = count_vec_g.get_feature_names_out()

count_cols_g = ['w_' + f for f in count_names_g]
count_df_g = pd.DataFrame(count_mat_g.toarray(), columns=count_cols_g)
count_df_g['date'] = articles_g['date'].values
count_monthly_g = count_df_g.groupby('date').sum().reset_index()

df_count_g = pd.merge(macro_g, count_monthly_g, on='date')
X_count_g = df_count_g[count_cols_g].values.astype(float)
sc_count_g = StandardScaler()
X_count_sc_g = sc_count_g.fit_transform(X_count_g)

print(f"TF-IDF monthly: {df_tfidf.shape[0]} months x {len(tfidf_cols)} terms")
print(f"Counts monthly: {df_count_g.shape[0]} months x {len(count_cols_g)} terms")

In [ ]:
# ── Lasso (k=5) + OLS helper ─────────────────────────────────────────────────
def lasso_k_ols_g(y, X_sc, cols, df_use, k=5):
    a_grid = np.logspace(-6, 1, 200)
    a_out, coefs, _ = lasso_path(X_sc, y, alphas=a_grid, max_iter=50000)
    nnz = np.sum(coefs != 0, axis=0)
    m = nnz == k
    idx = np.where(a_out == a_out[m].min())[0][0] if m.any() else np.argmin(np.abs(nnz - k))
    sel = [cols[j] for j in range(len(cols)) if coefs[j, idx] != 0]
    if not sel:
        return [], None, 0.0
    ols = sm.OLS(y, sm.add_constant(df_use[sel])).fit()
    return sel, ols, ols.rsquared

# ── Side-by-side comparison for mret and indprol1 ───────────────────────────
for outcome in ['mret', 'indprol1']:
    y_tf = df_tfidf[outcome].values
    y_ct = df_count_g[outcome].values

    sel_tf, ols_tf, r2_tf = lasso_k_ols_g(y_tf, X_tfidf_sc, tfidf_cols, df_tfidf, k=5)
    sel_ct, ols_ct, r2_ct = lasso_k_ols_g(y_ct, X_count_sc_g, count_cols_g, df_count_g, k=5)

    print("=" * 85)
    print(f"  {outcome.upper()}")
    print("=" * 85)
    print(f"  {'TF-IDF (k=5)':<40s}  {'Raw Counts (k=5)':<40s}")
    print(f"  R2 = {r2_tf:.4f}{'':<30s}  R2 = {r2_ct:.4f}")
    print(f"  {'-'*40}  {'-'*40}")

    n = max(len(sel_tf), len(sel_ct))
    for i in range(n):
        t_str, c_str = '', ''
        if i < len(sel_tf):
            nm = sel_tf[i][2:]
            co = ols_tf.params[sel_tf[i]]
            pv = ols_tf.pvalues[sel_tf[i]]
            sg = '***' if pv<0.01 else '**' if pv<0.05 else '*' if pv<0.1 else ''
            t_str = f'{nm:20s} {co:+8.4f} {sg}'
        if i < len(sel_ct):
            nm = sel_ct[i][2:]
            co = ols_ct.params[sel_ct[i]]
            pv = ols_ct.pvalues[sel_ct[i]]
            sg = '***' if pv<0.01 else '**' if pv<0.05 else '*' if pv<0.1 else ''
            c_str = f'{nm:20s} {co:+8.4f} {sg}'
        print(f"  {t_str:<40s}  {c_str:<40s}")
    print()

In [ ]:
# ── Full OLS summary for TF-IDF mret ─────────────────────────────────────────
y_mret_g = df_tfidf['mret'].values
sel_tf_mret, _, _ = lasso_k_ols_g(y_mret_g, X_tfidf_sc, tfidf_cols, df_tfidf, k=5)
rename_tf = {c: c[2:] for c in sel_tf_mret}
X_disp = sm.add_constant(df_tfidf[sel_tf_mret].rename(columns=rename_tf))
ols_disp = sm.OLS(y_mret_g, X_disp).fit()
print("TF-IDF OLS Summary (mret):")
print(ols_disp.summary())

In [ ]:
# ── Aggregate summary across all 53 outcomes ─────────────────────────────────
vol_cols_g = [c for c in macro_g.columns if c.endswith('_vol')]
all_outcomes_g = ['mret', 'vol', 'indpro', 'indprol1'] + vol_cols_g

tfidf_r2s, count_r2s = [], []
for outcome in all_outcomes_g:
    _, _, r2t = lasso_k_ols_g(df_tfidf[outcome].values, X_tfidf_sc, tfidf_cols, df_tfidf, k=5)
    _, _, r2c = lasso_k_ols_g(df_count_g[outcome].values, X_count_sc_g, count_cols_g, df_count_g, k=5)
    tfidf_r2s.append(r2t)
    count_r2s.append(r2c)

tfidf_r2s = np.array(tfidf_r2s)
count_r2s = np.array(count_r2s)

print("=" * 70)
print(f"  SUMMARY: TF-IDF vs. Raw Counts (k=5, all {len(all_outcomes_g)} outcomes)")
print("=" * 70)
print(f"  TF-IDF  mean R2: {tfidf_r2s.mean():.4f}  median: {np.median(tfidf_r2s):.4f}")
print(f"  Counts  mean R2: {count_r2s.mean():.4f}  median: {np.median(count_r2s):.4f}")
print(f"  TF-IDF > Counts: {(tfidf_r2s > count_r2s).sum()}/{len(all_outcomes_g)}")
print(f"  Counts > TF-IDF: {(count_r2s > tfidf_r2s).sum()}/{len(all_outcomes_g)}")
print(f"  Tied:            {(tfidf_r2s == count_r2s).sum()}/{len(all_outcomes_g)}")

## Part 2(a): LLM-Generated Topics for Market Return Analysis

Use Gemini API (`gemini-2.5-flash`) to generate 1-3 keyword topics per headline in batches of 50, aggregate monthly, then run the same Lasso (k=5) + OLS pipeline on `mret`.

In [ ]:
# ── Part 2(a): LLM Topic Generation ──────────────────────────────────────────
# Topics were generated using run_full_pipeline.py with the google-genai SDK.
# Model: gemini-2.5-flash, temperature=0.3, batches of 50 articles.
# All 10,200 articles processed in 27.5 min with 0 empty results.
# Results saved to articles_with_topics.parquet.

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df_check = pd.read_parquet('articles_with_topics.parquet')
print(f"Loaded {len(df_check)} articles with LLM-generated topics")
print(f"Empty topics: {(df_check['generated_topics'] == '').sum()}")
print(f"\nSample topics:")
for i in range(10):
    print(f"  [{i}] {df_check['generated_topics'].iloc[i]}")
    print(f"       Headline: {df_check['headline'].iloc[i][:80]}")
    print()

In [ ]:
# ── Step 3: Monthly Aggregation of LLM Topics ────────────────────────────────
from sklearn.feature_extraction.text import CountVectorizer

df_llm = pd.read_parquet('articles_with_topics.parquet')
df_llm['date'] = pd.to_datetime(df_llm['display_date']).dt.to_period('M').dt.to_timestamp()

llm_vec = CountVectorizer(stop_words='english')
llm_dtm = llm_vec.fit_transform(df_llm['generated_topics'])
llm_feat_names = llm_vec.get_feature_names_out()
llm_cols = ['llm_' + f for f in llm_feat_names]

llm_dtm_df = pd.DataFrame(llm_dtm.toarray(), columns=llm_cols)
llm_dtm_df['date'] = df_llm['date'].values
llm_monthly = llm_dtm_df.groupby('date').sum().reset_index()

macro_2a = pd.read_csv('macro.csv', parse_dates=['date'])
df_2a = pd.merge(macro_2a, llm_monthly, on='date')

print(f"LLM DTM: {llm_dtm.shape[1]} unique terms from {len(df_llm)} articles")
print(f"Monthly: {llm_monthly.shape[0]} months, merged: {df_2a.shape[0]} obs")
print(f"Date range: {df_2a['date'].min().date()} to {df_2a['date'].max().date()}")

In [ ]:
# ── Step 4: Lasso (k=5) + OLS on mret using LLM topics ──────────────────────
from sklearn.linear_model import lasso_path
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

# Standardize LLM features
X_llm = df_2a[llm_cols].values.astype(float)
scaler_llm = StandardScaler()
X_llm_scaled = scaler_llm.fit_transform(X_llm)

y_mret_2a = df_2a['mret'].values

# Lasso path
alphas_grid = np.logspace(-6, 1, 200)
alphas_out, coefs, _ = lasso_path(X_llm_scaled, y_mret_2a, alphas=alphas_grid, max_iter=50000)
n_nonzero = np.sum(coefs != 0, axis=0)

# Select alpha yielding exactly 5 non-zero coefficients
mask_5 = n_nonzero == 5
if mask_5.any():
    alpha_sel = alphas_out[mask_5].min()
    idx_sel = np.where(alphas_out == alpha_sel)[0][0]
else:
    idx_sel = np.argmin(np.abs(n_nonzero - 5))
    alpha_sel = alphas_out[idx_sel]

selected_mask = coefs[:, idx_sel] != 0
selected_llm_topics = [llm_cols[j] for j in range(len(llm_cols)) if selected_mask[j]]

# OLS with selected features
X_ols_2a = sm.add_constant(df_2a[selected_llm_topics])
ols_2a = sm.OLS(y_mret_2a, X_ols_2a).fit()

print("=" * 70)
print("  Part 2(a): LLM-Generated Topics → Lasso (k=5) → OLS on mret")
print("=" * 70)
print(f"  Alpha = {alpha_sel:.6f}")
print(f"  R2 = {ols_2a.rsquared:.4f}, Adj R2 = {ols_2a.rsquared_adj:.4f}")
print(f"  Selected LLM topics ({len(selected_llm_topics)}):")
for t in selected_llm_topics:
    nm = t[4:]  # strip 'llm_' prefix
    sig = '***' if ols_2a.pvalues[t] < 0.01 else '**' if ols_2a.pvalues[t] < 0.05 else '*' if ols_2a.pvalues[t] < 0.1 else ''
    print(f"    {nm:30s}  coef={ols_2a.params[t]:+10.6f}  p={ols_2a.pvalues[t]:.4f} {sig}")

print(f"\n── Comparison to Part 1(a) ──")
print(f"  Part 1(a) Topics R2 (k=5):      0.1079")
print(f"  Part 1(e) Raw Counts R2 (k=5):   0.1971")
print(f"  Part 2(a) LLM Topics R2 (k=5):   {ols_2a.rsquared:.4f}")

print("\n" + "=" * 70)
print(ols_2a.summary())

## Part 2(b): Prompt Engineering Experiments

Three experiments to evaluate how prompt design choices affect LLM-generated topic quality:
1. **Persona Experiment (2.b.i)**: Bull (optimistic) vs. Bear (pessimistic) system instructions on all 10,200 articles
2. **Temperature Experiment (2.b.ii)**: Low (0.1) vs. High (0.9) temperature on 20 articles, 3 runs each
3. **Lookahead Bias Experiment (2.b.iii)**: Basic vs. constrained prompt on 10 GFC-era articles (2007-2008)

### Part 2(b)(i): Persona Experiment -- Bull vs. Bear

System instructions were modified to add persona framing:
- **Bull**: "You are an overly optimistic investor who sees opportunity in every situation."
- **Bear**: "You are a deeply skeptical investor who sees risk and danger in market developments."

Each persona processed all 10,200 WSJ headlines using `gemini-2.5-flash` at temperature=0.3. Results were aggregated monthly and run through the same Lasso (k=5) + OLS pipeline on `mret`.

In [ ]:
# ── Part 2(b)(i): Persona Experiment -- Load and Analyze ─────────────────────
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import lasso_path
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

macro = pd.read_csv('macro.csv', parse_dates=['date'])

def persona_lasso_ols(parquet_file, persona_name):
    """Load persona parquet, build monthly DTM, run Lasso(k=5) + OLS on mret."""
    df = pd.read_parquet(parquet_file)
    df['date'] = pd.to_datetime(df['display_date']).dt.to_period('M').dt.to_timestamp()
    
    empties = (df['generated_topics'] == '').sum()
    print(f"\n{persona_name.upper()} persona: {len(df)} articles, {empties} empty ({100*empties/len(df):.1f}%)")
    
    vec = CountVectorizer(stop_words='english')
    dtm = vec.fit_transform(df['generated_topics'])
    feat_names = vec.get_feature_names_out()
    cols = ['llm_' + f for f in feat_names]
    
    dtm_df = pd.DataFrame(dtm.toarray(), columns=cols)
    dtm_df['date'] = df['date'].values
    monthly = dtm_df.groupby('date').sum().reset_index()
    
    merged = pd.merge(macro, monthly, on='date')
    print(f"  DTM: {dtm.shape[1]} unique terms, {merged.shape[0]} months merged")
    
    X = merged[cols].values.astype(float)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    y = merged['mret'].values
    
    alphas_grid = np.logspace(-6, 1, 200)
    alphas_out, coefs, _ = lasso_path(X_scaled, y, alphas=alphas_grid, max_iter=50000)
    n_nonzero = np.sum(coefs != 0, axis=0)
    
    mask_5 = n_nonzero == 5
    if mask_5.any():
        alpha_sel = alphas_out[mask_5].min()
        idx_sel = np.where(alphas_out == alpha_sel)[0][0]
    else:
        idx_sel = np.argmin(np.abs(n_nonzero - 5))
        alpha_sel = alphas_out[idx_sel]
        print(f"  (Closest k={n_nonzero[idx_sel]})")
    
    selected_mask = coefs[:, idx_sel] != 0
    selected = [cols[j] for j in range(len(cols)) if selected_mask[j]]
    
    X_ols = sm.add_constant(merged[selected])
    ols = sm.OLS(y, X_ols).fit()
    
    print(f"  Alpha = {alpha_sel:.6f}")
    print(f"  R2 = {ols.rsquared:.4f}, Adj R2 = {ols.rsquared_adj:.4f}")
    print(f"  Selected topics ({len(selected)}):")
    for t in selected:
        nm = t[4:]
        sig = '***' if ols.pvalues[t] < 0.01 else '**' if ols.pvalues[t] < 0.05 else '*' if ols.pvalues[t] < 0.1 else ''
        print(f"    {nm:30s}  coef={ols.params[t]:+10.6f}  p={ols.pvalues[t]:.4f} {sig}")
    
    return ols.rsquared, ols.rsquared_adj, selected, ols

# Run analysis for each persona
import os
results = {}

for persona in ['bear', 'bull']:
    pfile = f'articles_with_topics_{persona}.parquet'
    if os.path.exists(pfile):
        r2, adj_r2, topics, ols = persona_lasso_ols(pfile, persona)
        results[persona] = {'r2': r2, 'adj_r2': adj_r2, 'topics': topics, 'ols': ols}
    else:
        print(f"\n{persona.upper()} parquet not found -- generation still in progress")

# Comparison table
print(f"\n{'='*70}")
print(f"  PERSONA COMPARISON: Lasso (k=5) + OLS on mret")
print(f"{'='*70}")
print(f"  {'Persona':<12} {'R2':>8} {'Adj R2':>8}  Selected Topics")
print(f"  {'-'*12} {'-'*8} {'-'*8}  {'-'*35}")
print(f"  {'Base':<12} {'0.1302':>8} {'0.1240':>8}  (from Part 2a)")
for persona, r in results.items():
    topic_str = ', '.join(t[4:] for t in r['topics'])
    print(f"  {persona.upper():<12} {r['r2']:8.4f} {r['adj_r2']:8.4f}  {topic_str}")

### Part 2(b)(ii): Temperature Experiment

20 random articles were processed 3 times each at temperature=0.1 (low randomness) and temperature=0.9 (high randomness). We measure output consistency across runs.

In [ ]:
# ── Part 2(b)(ii): Temperature Experiment Results ────────────────────────────
# Experiment run via run_temperature_experiment.py with gemini-2.5-flash
# 20 random articles (seed=42), 3 runs at temp=0.1 and 3 runs at temp=0.9

print("=" * 80)
print("  TEMPERATURE EXPERIMENT: Consistency of LLM Topic Generation")
print("=" * 80)

# Results from completed experiment
temp_results = {
    'temp_0.1': {
        'identical': 4,
        'total': 20,
        'avg_unique': 2.20,
        'description': 'Low temperature (more deterministic)'
    },
    'temp_0.9': {
        'identical': 1,
        'total': 20,
        'avg_unique': 2.60,
        'description': 'High temperature (more random)'
    }
}

print(f"\n  {'Setting':<35} {'Identical/20':>15} {'Avg Unique (of 3)':>20}")
print(f"  {'-'*35} {'-'*15} {'-'*20}")
for k, v in temp_results.items():
    print(f"  {v['description']:<35} {v['identical']:>7}/20 ({100*v['identical']/v['total']:.0f}%) {v['avg_unique']:>13.2f}")

print(f"\n  Key Findings:")
print(f"  - temp=0.1: 4/20 (20%) articles had identical output across all 3 runs")
print(f"  - temp=0.9: 1/20 (5%) articles had identical output across all 3 runs")
print(f"  - Average unique outputs per article: 2.20 (low) vs 2.60 (high)")
print(f"  - Even at low temperature, LLM outputs are NOT fully deterministic")
print(f"  - Higher temperature increases variability but the effect is moderate")
print(f"    because the 1-3 keyword format constrains output diversity")

print(f"\n  Sample comparisons (from experiment output):")
samples = [
    ("In Katrina's Wake: New Orleans Businesses...",
     ["Natural disaster, Business disruption"] * 3,
     ["Natural disaster, Business disruption"] * 3),
    ("Air Canada Picks 2 Finalists to Bid...",
     ["M&A activity, Strategic investment",
      "Corporate investment, Ownership change",
      "Corporate investment, Airline industry"],
     ["Corporate investment, Equity stake",
      "Corporate acquisition, Airline industry",
      "Airline M&A, Investment bids"]),
]
for headline, low_runs, high_runs in samples:
    print(f"\n  Headline: {headline}")
    for i, (l, h) in enumerate(zip(low_runs, high_runs)):
        marker_l = " " if i == 0 else "*"
        marker_h = " " if i == 0 else "*"
        print(f"    Run {i+1}: temp=0.1: {l:40s} | temp=0.9: {h}")

### Part 2(b)(iii): Lookahead Bias Experiment

10 articles from June 2007 - September 2008 (Global Financial Crisis era) were processed with two prompts:
- **Basic**: Standard financial analyst system instruction
- **Constrained**: Explicitly prohibits referencing future events (2008 crisis, Lehman Brothers, housing crash, TARP, etc.)

We check whether the basic prompt exhibits "lookahead bias" by using future-knowledge terms.

In [ ]:
# ── Part 2(b)(iii): Lookahead Bias Experiment Results ────────────────────────
# Experiment run via run_lookahead_experiment.py with gemini-2.5-flash
# 10 GFC-era articles (seed=123), basic vs constrained system prompt

print("=" * 100)
print("  LOOKAHEAD BIAS EXPERIMENT: Basic vs. Constrained Prompt (GFC-era articles)")
print("=" * 100)

# Results from completed experiment
lookahead_data = [
    ("2007-07-16", "Currency Trading -- Forex View: Dollar Seen Recovering...",
     "Currency fluctuation, Forex risk", "Currency volatility, Exchange rate risk", False, False),
    ("2007-08-24", "Commodities Report: Wheat Surges to 11-Year High...",
     "Commodity inflation, Supply risk, Food prices", "Commodity inflation, Supply chain risk, Food prices", False, False),
    ("2007-09-06", "Suitor's Pledge May Help Seal Deal to Buy Australia's Coles...",
     "M&A risk, Share price volatility", "M&A risk, Share price volatility", False, False),
    ("2007-10-17", "Helping Motorola Make the Most Of Its Razr Brand...",
     "Brand strategy, Market competition", "Brand value, Marketing effectiveness", False, False),
    ("2007-10-23", "It's Hard to Heed the Experts When Your Kid Has the Cold...",
     "Social trends, Public health", "Consumer health spending, Personal finance", False, False),
    ("2007-12-05", "Heard on the Street: With Blackstone Worries Old and New...",
     "Investment firm risk, Market sentiment", "Investment firm risk, Market sentiment", False, False),
    ("2008-04-17", "U.K.'s Brown Pushes Financial Overhaul in U.S. Visit...",
     "Financial regulation, Systemic risk", "Regulatory reform, Financial transparency, Systemic risk", True, True),
    ("2008-05-27", "Direction of U.K. Interest Rates in Doubt Amid Mixed Signals...",
     "Interest rate uncertainty, Monetary policy", "Interest rate uncertainty, Monetary policy", False, False),
    ("2008-05-06", "McCain Speech to Shed Light on Judicial Philosophy...",
     "Political uncertainty, Policy impact", "Political uncertainty, Policy outlook", False, False),
    ("2008-06-17", "Medicis to Buy LipoSonix in a Bet on Body Shaping...",
     "M&A activity, Corporate strategy", "Acquisition risk, Market speculation", False, False),
]

n_different = 0
basic_bias = 0
constrained_bias = 0

for date, headline, basic, constrained, b_flag, c_flag in lookahead_data:
    same = "SAME" if basic == constrained else "DIFFERENT"
    if basic != constrained:
        n_different += 1
    if b_flag:
        basic_bias += 1
    if c_flag:
        constrained_bias += 1
    
    b_marker = " !! LOOKAHEAD" if b_flag else ""
    c_marker = " !! LOOKAHEAD" if c_flag else ""
    print(f"\n  {date} -- {headline}")
    print(f"    Basic:       {basic}{b_marker}")
    print(f"    Constrained: {constrained}{c_marker}")
    print(f"    >>> {same}")

print(f"\n{'='*100}")
print(f"  SUMMARY")
print(f"{'='*100}")
print(f"  Basic prompt:       {basic_bias}/10 articles had lookahead-indicative keywords")
print(f"  Constrained prompt: {constrained_bias}/10 articles had lookahead-indicative keywords")
print(f"  Outputs that differed: {n_different}/10")

print(f"\n  Lookahead keywords checked: crash, crisis, lehman, collapse, recession, tarp,")
print(f"    bailout, meltdown, contagion, systemic, subprime, great recession, 2008, financial crisis")

print(f"\n  Interpretation:")
print(f"    - Both prompts showed minimal lookahead bias at the keyword level (1/10 each)")
print(f"    - The flagged term ('systemic') appeared in BOTH prompts for an April 2008 article")
print(f"      about UK financial overhaul -- arguably a reasonable in-context term, not bias")
print(f"    - 7/10 outputs differed, showing the constrained prompt does change generation")
print(f"    - The short 1-3 keyword format naturally limits the opportunity for lookahead bias")
print(f"    - Longer-form analysis (e.g., paragraph summaries) would likely show more bias")

## Part 2(c): Out-of-Sample Forecasting of Industrial Production Growth

**Question**: Using the LLM topic generation approach, how well can we forecast industrial production growth?

**Approach**:
- **Target**: `indprol1` -- 1-month-ahead industrial production growth (already a growth rate in `macro.csv`)
- **Features**: 2,025 LLM-generated topic terms from the base (neutral) prompt only
- **Model**: Ridge Regression with leave-one-out GCV for automatic alpha selection
- **Procedure**: Expanding window -- start with 204 months of training data (Jan 1984 - Dec 2000), predict each subsequent month by re-fitting Ridge on all available history up to that point. Features are standardized at each step using only training data (no look-ahead).

In [ ]:
# ── Part 2(c): OOS Forecasting of indprol1 with LLM Topics ──────────────────
import pandas as pd
import numpy as np
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

# -- 1. Build monthly LLM topic features from base/neutral prompt ---------
llm_articles = pd.read_parquet('articles_with_topics.parquet')
llm_articles['date'] = pd.to_datetime(llm_articles['display_date']).dt.to_period('M').dt.to_timestamp()

llm_vec = CountVectorizer(stop_words='english')
llm_dtm = llm_vec.fit_transform(llm_articles['generated_topics'])
llm_names = llm_vec.get_feature_names_out()
llm_cols = ['llm_' + f for f in llm_names]

# Efficient sparse aggregation to monthly
dates_llm = llm_articles['date'].values
unique_dates = np.sort(pd.unique(dates_llm))
date_to_idx = {d: i for i, d in enumerate(unique_dates)}
row_idx = np.array([date_to_idx[d] for d in dates_llm])
agg = sparse.csr_matrix((np.ones(len(dates_llm)), (row_idx, np.arange(len(dates_llm)))),
                          shape=(len(unique_dates), len(dates_llm)))
llm_monthly_arr = (agg @ llm_dtm).toarray()

# -- 2. Merge with macro target -------------------------------------------
macro = pd.read_csv('macro.csv', parse_dates=['date'])
monthly_df = pd.DataFrame(llm_monthly_arr, columns=llm_cols)
monthly_df['date'] = unique_dates
df = pd.merge(macro[['date', 'indprol1']], monthly_df, on='date').sort_values('date').reset_index(drop=True)
df = df[df['indprol1'].notna()].reset_index(drop=True)

X_all = df[llm_cols].values.astype(np.float64)
y_all = df['indprol1'].values
dates_final = df['date'].values

print(f"Dataset: {len(y_all)} months, {len(llm_cols)} LLM features")
print(f"Target: indprol1 (mean={y_all.mean():.6f}, std={y_all.std():.6f})")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

# -- 3. Expanding-window Ridge forecast ------------------------------------
min_train = int(len(y_all) * 0.50)  # First 50% for initial training
n_test = len(y_all) - min_train
alphas = np.logspace(0, 6, 15)

print(f"\nExpanding window: train starts with {min_train} months, {n_test} test months")
print(f"Test period: {pd.Timestamp(dates_final[min_train]).date()} to {pd.Timestamp(dates_final[-1]).date()}")

y_pred_oos = np.full(n_test, np.nan)
for i in range(n_test):
    t = min_train + i
    scaler = StandardScaler(with_mean=False)
    X_train_sc = scaler.fit_transform(X_all[:t])
    X_test_sc = scaler.transform(X_all[t:t+1])
    ridge = RidgeCV(alphas=alphas, gcv_mode='svd')
    ridge.fit(X_train_sc, y_all[:t])
    y_pred_oos[i] = ridge.predict(X_test_sc)[0]

y_actual = y_all[min_train:]
oos_r2 = r2_score(y_actual, y_pred_oos)
oos_mse = np.mean((y_actual - y_pred_oos) ** 2)

# Historical mean baseline
hist_mean_pred = np.array([y_all[:min_train+i].mean() for i in range(n_test)])
r2_hist_mean = r2_score(y_actual, hist_mean_pred)

print(f"\n{'='*70}")
print(f"  OOS FORECASTING RESULTS: indprol1 (Industrial Production Growth)")
print(f"{'='*70}")
print(f"  Model:          Ridge Regression (LOO-GCV alpha selection)")
print(f"  Features:       {len(llm_cols)} LLM-generated topic terms")
print(f"  Test period:    {pd.Timestamp(dates_final[min_train]).date()} to {pd.Timestamp(dates_final[-1]).date()} ({n_test} months)")
print(f"")
print(f"  OOS R-squared (Ridge + LLM topics): {oos_r2:+.4f}")
print(f"  OOS R-squared (historical mean):    {r2_hist_mean:+.4f}")
print(f"  OOS MSE:                            {oos_mse:.8f}")
print(f"{'='*70}")

print(f"\n  Interpretation:")
print(f"  - The OOS R2 of {oos_r2:.4f} is negative, meaning the model does not beat")
print(f"    a simple constant (test-set mean) predictor out of sample.")
print(f"  - However, it slightly outperforms the expanding historical mean ({r2_hist_mean:.4f}),")
print(f"    suggesting the LLM topics contain a small marginal signal.")
print(f"  - This is a common result in macro forecasting: text features that show")
print(f"    strong in-sample R2 (Part 2a: 0.1302) often fail to generalize OOS")
print(f"    because the signal-to-noise ratio is low and patterns shift over time.")
print(f"  - The high Ridge alpha (~2,700-7,200) indicates the model needs extreme")
print(f"    regularization, confirming that most of the 2,025 topic features are noise.")

## Part 3(a): Embedding-Based Prediction of Market Returns

**Approach**: Generate 250-dimensional embeddings for each headline using Google Gemini (`gemini-embedding-001`), aggregate to monthly means, then run the same Lasso (k=5) + OLS pipeline on `mret`.

**Key Insight**: Dense embeddings compress all semantic information into 250 continuous dimensions, unlike the sparse topic/count representations (180-18K discrete features). With only k=5 non-zero coefficients, Lasso must select just 5 of these 250 dimensions — artificially constraining the dense representation. As we show below, embeddings dramatically improve when allowed more features (k=14-19), suggesting they encode richer information that requires more dimensions to express.

In [ ]:
# ── Part 3(a): Embeddings → Lasso (k=5) → OLS on mret ────────────────────────
import pandas as pd
import numpy as np
from sklearn.linear_model import lasso_path
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# -- 1. Load embeddings and aggregate to monthly mean ----------------------
df_emb = pd.read_parquet('articles_with_embeddings.parquet')
embeddings = np.array(df_emb['embedding'].tolist())
df_emb['date'] = pd.to_datetime(df_emb['display_date']).dt.to_period('M').dt.to_timestamp()

emb_cols = [f'emb_{i}' for i in range(embeddings.shape[1])]
emb_df = pd.DataFrame(embeddings, columns=emb_cols)
emb_df['date'] = df_emb['date'].values
emb_monthly = emb_df.groupby('date').mean().reset_index()

print(f"Loaded {len(df_emb)} articles with {embeddings.shape[1]}-dim embeddings")
print(f"Monthly aggregation (mean): {emb_monthly.shape[0]} months")

# -- 2. Merge with macro --------------------------------------------------
macro = pd.read_csv('macro.csv', parse_dates=['date'])
df_3a = pd.merge(macro[['date', 'mret']], emb_monthly, on='date').sort_values('date').reset_index(drop=True)

X_emb = df_3a[emb_cols].values.astype(np.float64)
y_mret = df_3a['mret'].values
scaler = StandardScaler()
X_emb_scaled = scaler.fit_transform(X_emb)

print(f"Merged: {df_3a.shape[0]} months, date range: {df_3a['date'].min().date()} to {df_3a['date'].max().date()}")

# -- 3. Lasso path → select k=5 → OLS ------------------------------------
alphas_grid = np.logspace(-6, 1, 200)
alphas_out, coefs, _ = lasso_path(X_emb_scaled, y_mret, alphas=alphas_grid, max_iter=50000)
n_nonzero = np.sum(coefs != 0, axis=0)

mask_5 = n_nonzero == 5
if mask_5.any():
    alpha_sel = alphas_out[mask_5].min()
    idx_sel = np.where(alphas_out == alpha_sel)[0][0]
else:
    idx_sel = np.argmin(np.abs(n_nonzero - 5))
    alpha_sel = alphas_out[idx_sel]

actual_k = n_nonzero[idx_sel]
selected_mask = coefs[:, idx_sel] != 0
selected_cols = [emb_cols[j] for j in range(len(emb_cols)) if selected_mask[j]]

X_ols = sm.add_constant(df_3a[selected_cols])
ols_3a = sm.OLS(y_mret, X_ols).fit()

print(f"\n{'='*70}")
print(f"  RESULTS: Embeddings Lasso (k={actual_k}) + OLS on mret")
print(f"{'='*70}")
print(f"  Alpha = {alpha_sel:.6f}, R2 = {ols_3a.rsquared:.4f}, Adj R2 = {ols_3a.rsquared_adj:.4f}")
print(f"  F-stat = {ols_3a.fvalue:.2f} (p = {ols_3a.f_pvalue:.2e})")
print(f"\n  Selected dimensions ({len(selected_cols)}):")
for c in selected_cols:
    sig = '***' if ols_3a.pvalues[c] < 0.01 else '**' if ols_3a.pvalues[c] < 0.05 else '*' if ols_3a.pvalues[c] < 0.1 else ''
    print(f"    {c:10s}  coef={ols_3a.params[c]:+12.6f}  p={ols_3a.pvalues[c]:.4f} {sig}")

# -- 4. Comparison table ---------------------------------------------------
print(f"\n{'='*70}")
print(f"  COMPARISON: All text representations, Lasso (k=5) + OLS on mret")
print(f"{'='*70}")
print(f"  {'Representation':<35} {'# Features':>10} {'R2':>8}")
print(f"  {'-'*35} {'-'*10} {'-'*8}")
print(f"  {'Part 1(a) Pre-built Topics':<35} {'180':>10} {'0.1079':>8}")
print(f"  {'Part 1(e) Raw Word Counts':<35} {'18,093':>10} {'0.1971':>8}")
print(f"  {'Part 1(g) TF-IDF':<35} {'18,093':>10} {'0.1811':>8}")
print(f"  {'Part 2(a) LLM-Gen Topics':<35} {'2,025':>10} {'0.1302':>8}")
print(f"  {'Part 3(a) Embeddings':<35} {'250':>10} {ols_3a.rsquared:8.4f}")

# -- 5. Embeddings across different k values (key insight) -----------------
print(f"\n  Embeddings R2 by number of selected dimensions:")
print(f"  {'k':>5} {'R2':>8} {'Adj R2':>8}  Note")
print(f"  {'-'*5} {'-'*8} {'-'*8}  {'-'*30}")
for k in [3, 5, 7, 10, 14, 19, 25, 30]:
    mk = n_nonzero == k
    if mk.any():
        a_k = alphas_out[mk].min()
        i_k = np.where(alphas_out == a_k)[0][0]
    else:
        i_k = np.argmin(np.abs(n_nonzero - k))
    sel_k = coefs[:, i_k] != 0
    actual = sum(sel_k)
    sel_cols_k = [emb_cols[j] for j in range(len(emb_cols)) if sel_k[j]]
    if sel_cols_k:
        ols_k = sm.OLS(y_mret, sm.add_constant(df_3a[sel_cols_k])).fit()
        note = ""
        if actual == 5: note = "← matches topic-based k"
        if actual == 14: note = "← surpasses raw counts (0.1971)"
        if actual == 19: note = "← best embedding model"
        print(f"  {actual:5d} {ols_k.rsquared:8.4f} {ols_k.rsquared_adj:8.4f}  {note}")

print(f"\n{'='*70}")
print(f"  KEY INSIGHT: At k=5, embeddings (R2={ols_3a.rsquared:.4f}) are comparable to")
print(f"  pre-built topics (0.1079). But embeddings scale far better with k:")
print(f"  at k~14, they surpass raw counts (0.1971), and at k~19 they reach")
print(f"  R2~0.22 — the BEST of any representation. Dense embeddings encode")
print(f"  richer semantic information that cannot be captured in just 5 dims.")
print(f"{'='*70}")

## Part 3(b): Topic Recovery — Can Embeddings Reconstruct Pre-Built Topics?

**Question**: The 250-dim embedding is a "black box" — do these dimensions actually encode interpretable topic information?

**Method**: For each of the 180 pre-built topics in `topics.csv`, regress the monthly topic attention score (Y) on the 250 monthly embedding features (X) using Ridge regression (GCV alpha). If embeddings truly capture topic content, we should see high R2 values — the embedding dimensions can linearly reconstruct the topic signals.

## Part 3(c): Topic Recovery — LLM-Generated Topics

Same approach applied to the 2,025 LLM-generated topic terms from Part 2(a). Since these are noisier (CountVectorizer on short keyword phrases), we expect lower recovery rates.

In [ ]:
# ── Part 3(b): Recovering Pre-Built Topics from Embeddings ───────────────────
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

# -- Load monthly embeddings (reuse from 3a if available) ------------------
df_emb = pd.read_parquet('articles_with_embeddings.parquet')
embeddings = np.array(df_emb['embedding'].tolist())
df_emb['date'] = pd.to_datetime(df_emb['display_date']).dt.to_period('M').dt.to_timestamp()

emb_cols = [f'emb_{i}' for i in range(embeddings.shape[1])]
emb_df = pd.DataFrame(embeddings, columns=emb_cols)
emb_df['date'] = df_emb['date'].values
emb_monthly = emb_df.groupby('date').mean().reset_index()

# -- Merge with pre-built topics -------------------------------------------
topics = pd.read_csv('topics.csv', parse_dates=['date'])
topic_cols = [c for c in topics.columns if c != 'date']
df_3b = pd.merge(topics, emb_monthly, on='date').sort_values('date').reset_index(drop=True)

X_emb_3b = df_3b[emb_cols].values.astype(np.float64)
scaler_3b = StandardScaler()
X_emb_3b_sc = scaler_3b.fit_transform(X_emb_3b)

print(f"Part 3(b): {df_3b.shape[0]} months, {len(topic_cols)} topics, {len(emb_cols)} embedding dims")

# -- Ridge regression: Y=topic, X=embeddings for each topic ----------------
alphas = np.logspace(-2, 4, 50)
results_3b = []

for tc in topic_cols:
    y = df_3b[tc].values
    ridge = RidgeCV(alphas=alphas, gcv_mode='svd')
    ridge.fit(X_emb_3b_sc, y)
    y_pred = ridge.predict(X_emb_3b_sc)
    r2 = r2_score(y, y_pred)
    results_3b.append({'topic': tc, 'r2': r2, 'alpha': ridge.alpha_})

results_3b.sort(key=lambda x: x['r2'], reverse=True)

# -- Print results ---------------------------------------------------------
print(f"\n{'='*70}")
print(f"  Part 3(b): Pre-Built Topic Recovery from 250-dim Embeddings")
print(f"{'='*70}")

print(f"\n  TOP 15 BEST RECOVERED:")
print(f"  {'Topic':<40} {'R2':>8}")
print(f"  {'-'*40} {'-'*8}")
for r in results_3b[:15]:
    bar = '#' * int(r['r2'] * 25)
    print(f"  {r['topic']:<40} {r['r2']:8.4f}  {bar}")

print(f"\n  BOTTOM 10 WORST RECOVERED:")
print(f"  {'Topic':<40} {'R2':>8}")
print(f"  {'-'*40} {'-'*8}")
for r in results_3b[-10:]:
    bar = '#' * int(r['r2'] * 25)
    print(f"  {r['topic']:<40} {r['r2']:8.4f}  {bar}")

r2_vals_3b = [r['r2'] for r in results_3b]
print(f"\n  SUMMARY (all {len(topic_cols)} topics):")
print(f"    Mean R2:   {np.mean(r2_vals_3b):.4f}")
print(f"    Median R2: {np.median(r2_vals_3b):.4f}")
print(f"    R2 > 0.5:  {sum(1 for r in r2_vals_3b if r > 0.5)}/{len(topic_cols)} ({100*sum(1 for r in r2_vals_3b if r > 0.5)/len(topic_cols):.0f}%)")
print(f"    R2 > 0.7:  {sum(1 for r in r2_vals_3b if r > 0.7)}/{len(topic_cols)} ({100*sum(1 for r in r2_vals_3b if r > 0.7)/len(topic_cols):.0f}%)")
print(f"    R2 > 0.9:  {sum(1 for r in r2_vals_3b if r > 0.9)}/{len(topic_cols)}")

# Featured topics
print(f"\n  NOTABLE TOPICS:")
for tname in ['Recession', 'Oil market', 'Federal Reserve', 'Financial crisis', 'Elections', 'Terrorism', 'China', 'Treasury bonds']:
    match = [r for r in results_3b if r['topic'] == tname]
    if match:
        r = match[0]
        rank = results_3b.index(r) + 1
        print(f"    {r['topic']:<30s}  R2 = {r['r2']:.4f}  (rank {rank}/{len(topic_cols)})")

print(f"\n{'='*70}")
print(f"  CONCLUSION: Embeddings can successfully reconstruct most pre-built topics.")
print(f"  131/180 (73%) topics are recovered with R2 > 0.5, meaning the 250-dim")
print(f"  embedding space encodes most of the same information as the hand-built")
print(f"  topic attention scores. Specific, event-driven topics (Financial crisis,")
print(f"  Terrorism, Recession) are recovered well, while generic/rare topics")
print(f"  (Credit cards, Buffett, Systems) are harder to reconstruct.")
print(f"{'='*70}")

In [ ]:
# ── Part 3(c): Recovering LLM-Generated Topics from Embeddings ───────────────
from sklearn.feature_extraction.text import CountVectorizer
from scipy import sparse

# -- Build monthly LLM topic counts ---------------------------------------
llm_articles = pd.read_parquet('articles_with_topics.parquet')
llm_articles['date'] = pd.to_datetime(llm_articles['display_date']).dt.to_period('M').dt.to_timestamp()

llm_vec = CountVectorizer(stop_words='english')
llm_dtm = llm_vec.fit_transform(llm_articles['generated_topics'])
llm_names = llm_vec.get_feature_names_out()
llm_cols_3c = ['llm_' + f for f in llm_names]

# Sparse aggregation to monthly
dates_llm = llm_articles['date'].values
unique_dates = np.sort(pd.unique(dates_llm))
date_to_idx = {d: i for i, d in enumerate(unique_dates)}
row_idx = np.array([date_to_idx[d] for d in dates_llm])
agg = sparse.csr_matrix((np.ones(len(dates_llm)), (row_idx, np.arange(len(dates_llm)))),
                          shape=(len(unique_dates), len(dates_llm)))
llm_monthly_arr = (agg @ llm_dtm).toarray()
llm_monthly_df = pd.DataFrame(llm_monthly_arr, columns=llm_cols_3c)
llm_monthly_df['date'] = unique_dates

# Merge with embeddings
df_3c = pd.merge(llm_monthly_df, emb_monthly, on='date').sort_values('date').reset_index(drop=True)
X_emb_3c = df_3c[emb_cols].values.astype(np.float64)
scaler_3c = StandardScaler()
X_emb_3c_sc = scaler_3c.fit_transform(X_emb_3c)

print(f"Part 3(c): {df_3c.shape[0]} months, {len(llm_names)} LLM terms, {len(emb_cols)} embedding dims")

# -- Ridge regression for top 50 most frequent LLM terms -------------------
total_freq = llm_monthly_arr.sum(axis=0)
top50_idx = total_freq.argsort()[::-1][:50]
top50_llm_cols = [llm_cols_3c[i] for i in top50_idx]

results_3c = []
for lc in top50_llm_cols:
    y = df_3c[lc].values.astype(float)
    ridge = RidgeCV(alphas=alphas, gcv_mode='svd')
    ridge.fit(X_emb_3c_sc, y)
    y_pred = ridge.predict(X_emb_3c_sc)
    r2 = r2_score(y, y_pred)
    freq = total_freq[llm_cols_3c.index(lc)]
    results_3c.append({'term': lc[4:], 'r2': r2, 'freq': freq})

results_3c.sort(key=lambda x: x['r2'], reverse=True)

print(f"\n{'='*70}")
print(f"  Part 3(c): LLM Topic Recovery from 250-dim Embeddings")
print(f"{'='*70}")

print(f"\n  TOP 15 BEST RECOVERED (of top-50 most frequent LLM terms):")
print(f"  {'LLM Term':<25} {'Freq':>6} {'R2':>8}")
print(f"  {'-'*25} {'-'*6} {'-'*8}")
for r in results_3c[:15]:
    bar = '#' * int(r['r2'] * 25)
    print(f"  {r['term']:<25} {r['freq']:6.0f} {r['r2']:8.4f}  {bar}")

print(f"\n  BOTTOM 10 WORST RECOVERED:")
print(f"  {'LLM Term':<25} {'Freq':>6} {'R2':>8}")
print(f"  {'-'*25} {'-'*6} {'-'*8}")
for r in results_3c[-10:]:
    bar = '#' * int(r['r2'] * 25)
    print(f"  {r['term']:<25} {r['freq']:6.0f} {r['r2']:8.4f}  {bar}")

r2_vals_3c = [r['r2'] for r in results_3c]
print(f"\n  SUMMARY (top 50 LLM terms):")
print(f"    Mean R2:   {np.mean(r2_vals_3c):.4f}")
print(f"    Median R2: {np.median(r2_vals_3c):.4f}")
print(f"    R2 > 0.5:  {sum(1 for r in r2_vals_3c if r > 0.5)}/{len(results_3c)}")
print(f"    R2 > 0.7:  {sum(1 for r in r2_vals_3c if r > 0.7)}/{len(results_3c)}")

# -- Overall comparison ----------------------------------------------------
print(f"\n{'='*70}")
print(f"  OVERALL: Can Embeddings Reconstruct Topic Signals?")
print(f"{'='*70}")
print(f"  Pre-built topics (180 topics from topics.csv):")
print(f"    Mean R2 = {np.mean(r2_vals_3b):.4f}, Median = {np.median(r2_vals_3b):.4f}")
print(f"    {sum(1 for r in r2_vals_3b if r > 0.5)}/{len(r2_vals_3b)} topics recovered with R2 > 0.5 (73%)")
print(f"")
print(f"  LLM-generated topics (top 50 of 2,025 terms):")
print(f"    Mean R2 = {np.mean(r2_vals_3c):.4f}, Median = {np.median(r2_vals_3c):.4f}")
print(f"    {sum(1 for r in r2_vals_3c if r > 0.5)}/{len(r2_vals_3c)} terms recovered with R2 > 0.5 (24%)")
print(f"")
print(f"  Interpretation:")
print(f"  - YES, embeddings successfully encode topic-level information.")
print(f"  - Pre-built topics are well-recovered (mean R2=0.62) because they are")
print(f"    carefully constructed, broad attention measures with smooth time series.")
print(f"  - LLM topics are harder to recover (mean R2=0.36) because they are noisy")
print(f"    keyword counts from short phrases, with more sparsity and measurement error.")
print(f"  - Event-driven topics (Recession R2=0.74, Financial crisis R2=0.83,")
print(f"    Terrorism R2=0.76) are especially well-captured by embeddings, confirming")
print(f"    that the semantic embedding space encodes newsworthy event information.")
print(f"{'='*70}")

## Part 3(d): OOS Forecasting of Industrial Production Growth with Embeddings

**Question**: Can 250-dimensional embeddings forecast `indprol1` better than the 2,025 LLM topic features from Part 2(c)?

**Setup**: Identical to Part 2(c) — expanding-window Ridge regression with LOO-GCV alpha selection. Initial training window = 204 months (Jan 1984 – Dec 2000), test period = 204 months (Jan 2001 – Dec 2017). Features standardized at each step using only training data.

In [ ]:
# ── Part 3(d): OOS Forecasting of indprol1 with Embeddings ───────────────────
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

# -- 1. Build monthly embedding features ----------------------------------
df_emb = pd.read_parquet('articles_with_embeddings.parquet')
embeddings = np.array(df_emb['embedding'].tolist())
df_emb['date'] = pd.to_datetime(df_emb['display_date']).dt.to_period('M').dt.to_timestamp()

emb_cols = [f'emb_{i}' for i in range(embeddings.shape[1])]
emb_df = pd.DataFrame(embeddings, columns=emb_cols)
emb_df['date'] = df_emb['date'].values
emb_monthly = emb_df.groupby('date').mean().reset_index()

# -- 2. Merge with macro --------------------------------------------------
macro = pd.read_csv('macro.csv', parse_dates=['date'])
df = pd.merge(macro[['date', 'indprol1']], emb_monthly, on='date').sort_values('date').reset_index(drop=True)
df = df[df['indprol1'].notna()].reset_index(drop=True)

X_all = df[emb_cols].values.astype(np.float64)
y_all = df['indprol1'].values
dates_final = df['date'].values
n_total = len(y_all)

print(f"Dataset: {n_total} months, {len(emb_cols)} embedding features")
print(f"Target: indprol1 (mean={y_all.mean():.6f}, std={y_all.std():.6f})")

# -- 3. Expanding-window Ridge forecast ------------------------------------
min_train = int(n_total * 0.50)
n_test = n_total - min_train
alphas = np.logspace(0, 6, 15)

print(f"Expanding window: {min_train} initial train, {n_test} test months")
print(f"Test period: {pd.Timestamp(dates_final[min_train]).date()} to {pd.Timestamp(dates_final[-1]).date()}")

y_pred_oos = np.full(n_test, np.nan)
selected_alphas = []

for i in range(n_test):
    t = min_train + i
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_all[:t])
    X_test_sc = scaler.transform(X_all[t:t+1])
    ridge = RidgeCV(alphas=alphas, gcv_mode='svd')
    ridge.fit(X_train_sc, y_all[:t])
    y_pred_oos[i] = ridge.predict(X_test_sc)[0]
    selected_alphas.append(ridge.alpha_)

# -- 4. Results ------------------------------------------------------------
y_actual = y_all[min_train:]
oos_r2 = r2_score(y_actual, y_pred_oos)
oos_mse = np.mean((y_actual - y_pred_oos) ** 2)

# Historical mean baseline
hist_mean_pred = np.array([y_all[:min_train+i].mean() for i in range(n_test)])
r2_hist_mean = r2_score(y_actual, hist_mean_pred)

print(f"\n{'='*70}")
print(f"  OOS FORECASTING RESULTS: indprol1 (Industrial Production Growth)")
print(f"{'='*70}")
print(f"  {'Model':<40} {'OOS R2':>10}")
print(f"  {'-'*40} {'-'*10}")
print(f"  {'Embeddings (250 dims, Ridge+GCV)':<40} {oos_r2:+10.4f}")
print(f"  {'LLM Topics (2,025 terms, Part 2c)':<40} {'-0.0478':>10}")
print(f"  {'Historical Mean Baseline':<40} {r2_hist_mean:+10.4f}")
print(f"{'='*70}")
print(f"  Median Ridge alpha: {np.median(selected_alphas):.0f}")
print(f"  OOS MSE: {oos_mse:.8f}")
print(f"{'='*70}")

### Part 3 — Final Summary: Embeddings vs. Topics for Forecasting and Analysis

#### OOS Forecasting Comparison (`indprol1`, expanding-window Ridge)

| Model | Features | OOS R² |
|-------|----------|--------|
| **Embeddings (Part 3d)** | 250 dims | **-0.0469** |
| LLM Topics (Part 2c) | 2,025 terms | -0.0478 |
| Historical Mean Baseline | — | -0.0565 |

#### In-Sample Contemporaneous Comparison (`mret`, Lasso k=5 + OLS)

| Representation | Features | R² |
|----------------|----------|-----|
| Part 1(a) Pre-built Topics | 180 | 0.1079 |
| Part 1(e) Raw Word Counts | 18,093 | 0.1971 |
| Part 2(a) LLM-Gen Topics | 2,025 | 0.1302 |
| **Part 3(a) Embeddings (k=5)** | 250 | **0.1055** |
| **Part 3(a) Embeddings (k=14)** | 250 | **0.2009** |
| **Part 3(a) Embeddings (k=19)** | 250 | **0.2234** |

#### Topic Recovery (Parts 3b & 3c)

| Source | Mean R² | Topics with R² > 0.5 |
|--------|---------|----------------------|
| Pre-built topics (180) | 0.6176 | 131/180 (73%) |
| LLM-gen topics (top 50) | 0.3601 | 12/50 (24%) |

#### Key Findings

1. **OOS Forecasting**: Embeddings (R² = -0.0469) and LLM topics (R² = -0.0478) perform nearly identically for forecasting industrial production growth. Both slightly beat the historical mean baseline (-0.0565), but neither achieves positive OOS R². This is a well-known result in macro forecasting: text-based features that show strong contemporaneous relationships (in-sample R² of 0.10–0.22) typically fail to generalize out-of-sample because the signal-to-noise ratio is too low and economic regimes shift over time.

2. **Embeddings are dimension-efficient**: With only 250 features (vs. 2,025 LLM terms or 18,093 word counts), embeddings achieve comparable or superior performance. At k=5 they match pre-built topics, and at k=14–19 they **surpass all other representations** for in-sample `mret` prediction. This is because dense embeddings distribute semantic information across all 250 dimensions, requiring more features to fully express their richness.

3. **Embeddings encode topic information**: Ridge regression can reconstruct 73% of pre-built topics (R² > 0.5) from the 250 embedding dimensions alone. Event-driven topics like Financial Crisis (0.83), Terrorism (0.76), and Recession (0.74) are recovered especially well, confirming that the embedding space captures the same economic concepts as hand-built topic attention scores.

4. **Why negative OOS R²?** The high Ridge alpha (~2,683) indicates extreme regularization is needed, confirming that most embedding dimensions are noise for predicting `indprol1`. The expanding window trains on 204+ months but must predict a target with std=0.006 — a very weak signal. The model slightly outperforms a naive mean forecast, suggesting a marginal but not economically significant signal exists in the text data.